In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Titanic Survival Prediction

This notebook predicts passenger survival on the Titanic using machine learning models  **XGBoost** and **LightGBM**.  
We prepare, preprocess, and blend multiple models to improve accuracy.


## Data Preparation

In [ ]:
train =pd.read_csv("/kaggle/input/titanic/train.csv")
test=pd.read_csv("/kaggle/input/titanic/test.csv")


In [ ]:
train.head()

In [ ]:
gender_submission=pd.read_csv("/kaggle/input/titanic/gender_submission.csv")

In [ ]:
gender_submission.head()

In [ ]:
train.shape

In [ ]:
train.info()

As the column cabin has a lot of null value therefore we are dropping it .
 In Addition the name column does not seem to have much relation with the survived.

In [ ]:
new_train = train.drop(columns=["Cabin","Name"])

### Converting the sex and embarked columns to categorical value

In [ ]:
for x in new_train.index:
    if new_train.loc[x,'Sex']=='male':
        new_train.loc[x,'Sex']=1
    else:
        new_train.loc[x,'Sex']=0
new_train["Sex"]=new_train["Sex"].astype('int64')

In [ ]:
new_train["Embarked"].unique()

In [ ]:
new_train["Embarked"].isnull().sum()


In [ ]:
new_train[new_train["Embarked"].isna()]


Passengers from _C_ embarkment seemed to have more survivals and therefore we are assuming that the passenger having null values in _embarked_ would have been from _C_

In [ ]:
survival_by_port = new_train.groupby("Embarked")["Survived"].mean()

print(survival_by_port)

In [ ]:
new_train.loc[61,"Embarked"]="C"
new_train.loc[829,"Embarked"]="C"

In [ ]:
new_train["Embarked"] = new_train["Embarked"].map({'C': 1, 'Q': 2, 'S': 3})
new_train["Embarked"] = new_train["Embarked"].astype('int64')


In [ ]:
new_train.head()

In [ ]:
ticket_counts = new_train['Ticket'].value_counts()
new_train['TicketCount'] = new_train['Ticket'].map(ticket_counts)


In [ ]:
new_train['TicketCount'].head()

In [ ]:
new_train.head()

### Preparing Test data for prediction


In [ ]:
test_ticket_counts= test['Ticket'].value_counts()
test['TicketCount']= test['Ticket'].map(test_ticket_counts)

In [ ]:
test['Sex']=test["Sex"].map({'male':1,'female':0})
new_train["Sex"]=new_train["Sex"].astype('int64')

In [ ]:
test["Embarked"] = test["Embarked"].map({'C': 1, 'Q': 2, 'S': 3})
test["Embarked"] = test["Embarked"].astype('int64')

In [ ]:
test.head()

##  Model Preparation

We'll use the following features for training:
- Pclass
- Sex
- SibSp
- Parch
- TicketCount

The target variable is **Survived**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import pandas as pd

y = new_train["Survived"]
features = ["Pclass", "Sex", "SibSp", "Parch", "TicketCount"]

X = pd.get_dummies(new_train[features])
X_test = pd.get_dummies(test[features])

X_test = X_test.reindex(columns=X.columns, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=1)
lgb = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=1)

xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)

xgb_val_pred = xgb.predict_proba(X_val)[:, 1]
lgb_val_pred = lgb.predict_proba(X_val)[:, 1]

xgb_labels = (xgb_val_pred > 0.5).astype(int)
lgb_labels = (lgb_val_pred > 0.5).astype(int)

print("XGBoost Accuracy:", accuracy_score(y_val, xgb_labels))
print("LightGBM Accuracy:", accuracy_score(y_val, lgb_labels))




In [ ]:
xgb.fit(X, y)
lgb.fit(X, y)

xgb_pred = xgb.predict_proba(X_test)[:, 1]
lgb_pred = lgb.predict_proba(X_test)[:, 1]

##  Blending Models
We check accuracies of both the models and blend the models on the basis of their accuracy.
This  improves generalization by combining the strengths of both models.


In [ ]:
final_pred = ( 0.4 * xgb_pred + 0.6 * lgb_pred)

final_labels = (final_pred > 0.5).astype(int)

output = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': final_labels})
output.to_csv('submission_blend.csv', index=False)

In [ ]:
output.head()